# 79장 · 웹과 API 연결

과정 안내 질문을 입력하고 전송을 누르면 어떤 값이 보내질까요? 78장의 투두 화면에서 입력칸과 버튼을 구별했습니다. 이번에는 별도 과정 안내 사례를 사용합니다. 노트북에서 submit 이벤트와 HTTP 응답을 재현하며 화면에 넣을 문장을 선택합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 입력 이벤트에서 요청 본문 꺼내기

브라우저의 submit 처리기는 event.preventDefault()로 기본 페이지 이동을 막고 입력값의 양끝을 정리합니다. 노트북에서는 Node.js가 설치된 수업 커널에서 실제 JavaScript를 실행합니다. DOM을 띄우지 않고 표준 EventTarget을 이용해 제출 동작만 재현합니다.



**볼 곳**　‘입력 이벤트’에서 출발해 ‘요청 본문’를 거친 뒤 ‘API’가 되는 순서를 따라가세요.

**애니메이션의 의미**　폼에 쓴 질문이 API 요청 본문에 들어갑니다.


### 셀 01 · 제공 셀 · JavaScript 실행 준비

코드를 바꿀 곳은 다음 셀의 JavaScript입니다. 제공 함수는 실행 결과를 노트북에 표시합니다.

In [1]:
import subprocess
def run_js(source):
    run = subprocess.run(["node", "--input-type=module", "-e", source],
                         text=True, capture_output=True, check=True, timeout=10)
    print(run.stdout.rstrip())

함수를 정의했습니다. 별도의 JS 파일을 만들 필요가 없습니다.

### 셀 02 · submit에서 질문을 수신하기

질문 양끝에 있는 공백이 JSON 본문에 남을지 예측하세요.

In [2]:
run_js("""
const form = new EventTarget();
const input = {value: "  과제는 어디에 내나요?  "};
form.addEventListener("submit", event => {
  event.preventDefault();
  const body = {question: input.value.trim(), k: 2};
  console.log(JSON.stringify(body));
});
form.dispatchEvent(new Event("submit", {cancelable: true}));
""")

{"question":"과제는 어디에 내나요?","k":2}


JSON에는 정리한 question과 검색 개수 k가 담겼습니다. 이 단계에서는 네트워크 요청이나 답변 표시가 일어나지 않았습니다.

## 응답이 도착한 뒤 본문 읽기

실제 화면의 transport.mjs는 fetch 함수를 전달받습니다. 이 셀에서는 수업용 응답을 주는 함수를 넣어 요청과 화면 사이를 재현합니다. 실제 서버 주소에서는 해당 함수 자리에 브라우저 fetch를 넣습니다.



**볼 곳**　‘응답 도착’에서 출발해 ‘본문 읽기’를 거친 뒤 ‘답 표시’가 되는 순서를 따라가세요.

**애니메이션의 의미**　응답이 도착한 뒤 본문에서 답과 출처를 읽습니다.


### 셀 03 · 응답의 A1 답변을 화면에 넣기

fetch 응답과 JSON 해석을 각각 await합니다. 여기서 element는 화면 요소의 textContent 속성만 흉내 낸 객체입니다.

In [3]:
run_js("""
const fixtureFetch = async (_url, options) => {
  const { question } = JSON.parse(options.body);
  if (question !== "과제는 어디에 내나요?") return new Response(null, {status: 422});
  return Response.json({result: {answer: "과제함에 제출하세요. [course-guide-v1:p2]"}});
};
const response = await fixtureFetch("/conversations/chat-a/ask", {
  method: "POST", headers: {"Content-Type":"application/json"},
  body: JSON.stringify({question:"과제는 어디에 내나요?", k:2})
});
if (!response.ok) throw Error("HTTP " + response.status);
const body = await response.json();
const element = {textContent: ""};
element.textContent = body.result.answer;
console.log(element.textContent);
""")

과제함에 제출하세요. [course-guide-v1:p2]


표시할 텍스트에는 출처 ID가 남았습니다. innerHTML로 문자열을 해석하지 않고 textContent에 넣으면 답에 꺾쇠가 있더라도 문구로 다룰 수 있습니다.

### 과제 04 · 실패 시 JSON 읽기 전에 분기하기

404 응답의 ok를 읽어 ‘대화를 찾을 수 없습니다’라는 문구를 선택하는 JavaScript를 작성하세요.

빈 본문을 돌려준 404에서는 JSON을 읽기 전에 멈춥니다. 저장 실패 503의 응답에는 이미 만든 답과 저장 경고가 남을 수 있습니다. 상태 코드를 확인한 뒤 그 API가 약속한 오류 본문 형식에 따라 읽으세요.

In [4]:
# run_js에 JavaScript를 전달하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>run_js(&#x27;const r = new Response(null,{status:404}); console.log(r.ok ? &quot;완료&quot; : &quot;대화를 찾을 수 없습니다&quot;);&#x27;)</code></pre>
<p>실행 결과</p><pre><code>대화를 찾을 수 없습니다</code></pre>
<p>404 응답은 Promise가 정상적으로 돌아올 수 있어도 ok는 false입니다. HTTP 오류와 연결 자체가 끊겨 Promise가 거절된 경우를 따로 처리합니다. 셀에서 응답을 준비한 곳은 Node입니다. 브라우저에서 확인할 때는 첫 POST 응답의 대화 ID를 다음 POST가 실제로 보내는지 개발자 도구의 네트워크 기록에서 대조하세요.</p>
</details>

## 돌아보기

같은 대화에서 이전 요청의 늦은 응답이 새 답변을 덮을 수 있습니다. 제출할 때 요청 번호를 기억하고 화면을 바꾸기 직전 현재 요청인지 확인할 위치를 찾아보세요.

### 한 대화의 기본 읽기 요청



**볼 곳**　첫 응답의 대화 ID가 다음 질문 요청의 경로에 들어가는 곳을 보세요.

**애니메이션의 의미**　서버가 만든 대화 ID를 다음 요청에 사용해야 질문과 응답이 같은 대화로 이어집니다.
